In [ ]:
import pickle as pk
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns          # if ImportError: pip install seaborn  (then restart the kernel)
from scipy.stats import spearmanr

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid", context="notebook")

FIG_DIR = Path("figures")          # figures for the progress report get saved here
FIG_DIR.mkdir(exist_ok=True)

# Week 2 Helper Notebook
This exploration should produce AT LEAST three figures , which you should include in the progress report and make look nice using seaborn (feel free to use AI to help with plotting). 
1. {AUROC/AUPRC} by held-out hospital, with internal {AUROC/AUPRC} as a dashed reference.
2. Generalization gap by each heldout hospital.
3. At least one exploratory diagnostic, such as generalization gap vs missingness shift or feature-distribution shift.

General guiding goal: 
* Can we establish that some hospitals generalize much worse than others? Can we trace that failure to particular features / can information available across the training hospitals tell us ahead of time which features are risky?

## Step 1: Load eICU data

In [ ]:
feature_sets = pk.load(open('/Users/gabrielarm300/cs197_fall2026/data/feature_names.pkl', 'rb'))
patient_agg = pd.read_parquet('/Users/gabrielarm300/cs197_fall2026/data/clean_dataset.parquet')

print("Dataset shape:", patient_agg.shape)
patient_agg.head()

In [ ]:
# This distinction of numerical (continuous), binary, and categorical
# columns may be useful throughout the project.
all_num_cols = (
    feature_sets['labs'].tolist()
    + feature_sets['vitals'].tolist()
    + ['age', 'admissionheight', 'admissionweight']
)

all_bin_cols = (
    feature_sets['icd10_before24h'].tolist()
    + feature_sets['admissiondx'].tolist()
)

all_cat_cols = [
    # 'hospital_region',
    'ethnicity',
    'gender',
    'hospital_numbedscategory',
    'hospitaldischargeyear',
    # 'hospitalid'
]

In [ ]:
# For privacy, eICU records patients older than 89 using the string "> 89".
if 'age' in patient_agg.columns:
    patient_agg.loc[patient_agg.age == '> 89', 'age'] = 90
    patient_agg['age'] = patient_agg['age'].astype(float)

## Step 1: Investigate possible prediction outcomes Y

In [ ]:
# Y must occur AFTER the time window used to construct X. Why?
#   X is built from the first 24 hours. If Y happened inside (or before) that window, X could "see" the
#   outcome (e.g. a test ordered *because* the patient was already crashing) and the model would look
#   great for the wrong reason (label leakage). mortality_at48h is measured after the 24h feature window.
#   (Double check in the Week 1 notebook that this is how it was defined.)
OUTCOME_COL = 'mortality_at48h'

# a missing outcome is treated as "did not happen" (same assumption as Week 1)
patient_agg[OUTCOME_COL] = patient_agg[OUTCOME_COL].fillna(0).astype(int)

print(f"Prevalence: {patient_agg[OUTCOME_COL].mean():.2%}")
patient_agg[OUTCOME_COL].value_counts(dropna=False)

## Step 2: Choose X
Think: What are variables that are realistic and useful in predicting Y? For example, we probably wouldn't use region of hospital to predict mortality, but we might use lab measurements.

In [ ]:
# Realistic predictors available inside the first 24h:
#   numeric:     labs, vitals, age, height, weight
#   binary:      diagnoses before 24h, admission diagnoses
#   categorical: gender, ethnicity   (ethnicity is a sensitive covariate -- drop it if you'd rather not use it)
#
# Deliberately NOT used:
#   hospitalid -> an identifier, and it is what we split on
#   hospital_region / hospital_numbedscategory / hospitaldischargeyear -> descriptions of the hospital,
#     not of the patient. A model using them can just memorise hospitals, which defeats the generalization question.
X_num = [c for c in all_num_cols if c in patient_agg.columns]
X_bin = [c for c in all_bin_cols if c in patient_agg.columns]
X_cat = [c for c in ['gender', 'ethnicity'] if c in patient_agg.columns]

X_cols = X_num + X_bin + X_cat
assert 'hospitalid' not in X_cols
assert OUTCOME_COL not in X_cols
print(f"{len(X_num)} numeric, {len(X_bin)} binary, {len(X_cat)} categorical features")

## Step 3: Understand different hospital statistics

In [ ]:
# Sample size, outcome prevalence, and a couple of other statistics for every hospital
hosp_stats = (
    patient_agg.groupby('hospitalid')
    .agg(n_patients=(OUTCOME_COL, 'size'),
         n_pos=(OUTCOME_COL, 'sum'),
         prevalence=(OUTCOME_COL, 'mean'),
         mean_age=('age', 'mean'))
)
hosp_stats['n_neg'] = hosp_stats['n_patients'] - hosp_stats['n_pos']

# average fraction of numeric features that are missing at each hospital
hosp_stats['avg_frac_missing'] = (
    patient_agg[X_num].isna().groupby(patient_agg['hospitalid']).mean().mean(axis=1)
)

# hospital descriptors (constant within a hospital)
hosp_attr_cols = [c for c in ['hospital_region', 'hospital_numbedscategory'] if c in patient_agg.columns]
if hosp_attr_cols:
    hosp_stats = hosp_stats.join(patient_agg.groupby('hospitalid')[hosp_attr_cols].first())

print(f"{len(hosp_stats)} hospitals")
display(hosp_stats.describe().T)
hosp_stats.sort_values('prevalence').head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
sns.histplot(hosp_stats['n_patients'], bins=30, ax=axes[0]);  axes[0].set_xlabel('patients per hospital')
sns.histplot(hosp_stats['prevalence'], bins=30, ax=axes[1]);  axes[1].set_xlabel(f'{OUTCOME_COL} prevalence')
sns.scatterplot(data=hosp_stats, x='n_patients', y='prevalence', ax=axes[2])
axes[2].set_xlabel('patients per hospital'); axes[2].set_ylabel('prevalence')
plt.tight_layout()
plt.show()

**What does it mean if a hospital has a really low Y prevalence, or a really small sample size? Should we remove them?**

*(draft answer, edit in your own words)*
- With few patients or few positive cases, metrics like AUROC/AUPRC are very noisy (one extra death can move them a lot), and if a hospital has **zero** positives they are undefined.
- Very low prevalence can be real (a low-acuity hospital) or an artifact (deaths recorded differently, patients transferred out before the outcome). It is also a different "task" from a high-prevalence hospital.
- We should not delete them blindly: they are still useful as **training** data. But we should not use them as **held-out** hospitals, because we could not measure performance there reliably. So below, held-out hospitals must pass minimum-size and minimum-positives thresholds.

## Step 4: Explore / pick possible hospital train/heldout splits

In [ ]:
# Pick some K, maybe 10. Then try splitting by hospital region, number of beds, for K hospitals in the training data. 
# Suggested requirements:
#
# n_train_patients > 10,000
# 5 <= n_heldout_hospitals < 15
#
# PLUS enough positive and negative outcomes in every held-out hospital.
K_HELDOUT    = 10
MIN_HELD_N   = 300     # min patients in a held-out hospital
MIN_HELD_POS = 15      # min positives AND min negatives in a held-out hospital
# (if you get too few eligible hospitals, lower these)

eligible = hosp_stats[(hosp_stats.n_patients >= MIN_HELD_N)
                      & (hosp_stats.n_pos >= MIN_HELD_POS)
                      & (hosp_stats.n_neg >= MIN_HELD_POS)]
print(f"{len(eligible)} of {len(hosp_stats)} hospitals are eligible to be held out")
assert len(eligible) >= K_HELDOUT, "Not enough eligible hospitals: lower MIN_HELD_N / MIN_HELD_POS"

# candidate splits: a few random ones, plus "hold out hospitals from one region"
eligible_ids = eligible.index.to_numpy()
candidates = {}
for s in range(5):
    candidates[f'random_{s}'] = np.random.default_rng(s).choice(eligible_ids, size=K_HELDOUT, replace=False)

if 'hospital_region' in eligible.columns:
    for region, grp in eligible.groupby('hospital_region'):
        ids = grp.index.to_numpy()
        if len(ids) >= 5:
            candidates[f'region={region}'] = ids if len(ids) < 15 else np.random.default_rng(0).choice(ids, 14, replace=False)

def summarize_split(ids):
    ids = list(ids)
    n_held = hosp_stats.loc[ids, 'n_patients'].sum()
    pos_held = hosp_stats.loc[ids, 'n_pos'].sum()
    n_total, pos_total = hosp_stats['n_patients'].sum(), hosp_stats['n_pos'].sum()
    n_train = n_total - n_held
    return dict(n_heldout_hosp=len(ids), n_train=int(n_train), n_heldout=int(n_held),
                train_prev=(pos_total - pos_held) / n_train, heldout_prev=pos_held / n_held,
                min_pos_in_heldout_hosp=int(hosp_stats.loc[ids, 'n_pos'].min()))

split_table = pd.DataFrame({name: summarize_split(ids) for name, ids in candidates.items()}).T
split_table['valid'] = (split_table.n_train > 10_000) & split_table.n_heldout_hosp.between(5, 14)
split_table

In [ ]:
## Create train / heldout data split
#  DOUBLE check there is no overlap of heldout data with train data!!
CHOSEN_SPLIT = 'random_0'     # <- change to try other splits (e.g. 'region=South')
assert split_table.loc[CHOSEN_SPLIT, 'valid'], "This split does not meet the suggested requirements"

heldout_hospitals = list(candidates[CHOSEN_SPLIT])
is_heldout = patient_agg['hospitalid'].isin(heldout_hospitals)

train_df   = patient_agg[~is_heldout].copy()    #all patients in the training hospitals
heldout_df = patient_agg[is_heldout].copy()     #all patients in the held-out hospitals

# --- sanity checks: no hospital (hence no patient) appears on both sides, and nothing was lost
assert set(train_df['hospitalid']).isdisjoint(set(heldout_df['hospitalid']))
assert len(train_df) + len(heldout_df) == len(patient_agg)
print(f"train: {len(train_df):,} patients in {train_df.hospitalid.nunique()} hospitals | "
      f"held-out: {len(heldout_df):,} patients in {heldout_df.hospitalid.nunique()} hospitals")

# Internal validation set: a random 20% of the TRAINING hospitals' patients (same hospitals as training).
# We tune the model on it, and it is our "internal performance" reference.
train_inner, val_inner = train_test_split(
    train_df, test_size=0.2, random_state=SEED, stratify=train_df[OUTCOME_COL]
)
train_inner, val_inner = train_inner.copy(), val_inner.copy()
print(f"inner train: {len(train_inner):,} | internal validation: {len(val_inner):,}")

## DETOUR: Handling missing data 

You may have noticed that variables are often missing in our dataset. For example, age is missing for 0.0346% of all patients, the lab measurement of ANF/ANA is missing for 99.9%, and height is missing for 2%.

Categorical variables can be missing too. For example, if a patient's gender was not recorded.

However, missing values are easier to handle in categorical variables than in continuous ones, because when we one-hot encode the variable, we can simply add one more column for "missing". For example:

|Patient| Recorded gender  | female | male | missing |
|:--|------------------|:------:|:----:|:-------:|
|A| female           |   1    |  0   |    0    |
|B| male             |   0    |  1   |    0    |
|C| *(not recorded)* |   0    |  0   |    1    |

The model treats "missing" like any other category, so it can learn how common it is and even generate it in synthetic data.

The challenge with continuous variables is that we need a numerical input to our model, because NaNs are not valid inputs (a single NaN in a row makes the loss NaN, and then all the weights become NaN too).

**Your goal is to briefly explore and pick one way of handling missingness for continuous variables.** Note, how you define missing values will change how you choose to model continuous variables. Some suggestions are:


1. **Drop the patients** that contain ANY missing value. Simple, but you may throw away a lot of data. 
2. **Fill with a constant**, such as the mean or median (computed on the training set only!). Easy, but many patients now share exactly the same value and the data can lose some predictive power. Use `sklearn.impute.SimpleImputer`. 
4. **Fill with the median and add a "was it missing?" binary variable**, the same trick as for categorical variables. Again use median of training data.
    - Note if you do this: You will need to add on these extra "was missing" variable (# new binary cols = # continuous variables) to the list above of `bin_cols`
6. **Predict the missing value from the other columns** on the training data only (for example a regression, k-nearest neighbors, or a MICE-style imputer such as scikit-learn's `IterativeImputer`). If you use imputation, you are free to use AI to help you impute the dataframe above !!
7. **Multiple imputation.** Fill the gaps several different (random) ways to get several complete datasets, then use a different one in each epoch. If you use imputation, you are free to use AI to help you impute the dataframe above !!


For example, say you build some Imputer class. You would want to do 
```
imp = Imputer()
imp.fit(train_df numerical data)

imputed train = imp.transform(train_df numerical data)
imputed val = imp.transform(val_df numerical data)

```

**What this notebook does:** option 4 (median + "was it missing?" indicator), fit on the inner training split only. Why option 4? In ICU data a lab is often missing *because the clinician didn't think it was needed*, so "missing" itself carries signal, and the indicator lets the model use it. It is also cheap and easy to explain. (Compare with option 2 as an ablation if you have time.)

In [ ]:
# first for categorical data, just fill in the missing values with the string Missing. 
# -Ask yourself, why? What might this information preserve? 
#   Answer: "not recorded" is often informative (e.g. patients admitted in a hurry, or transferred, have
#   less complete demographics). A "Missing" category keeps that signal instead of throwing the patient away
#   or guessing a value. It is a constant, not a statistic learned from data, so it cannot leak across splits.
for d in (train_inner, val_inner, heldout_df):
    d[X_cat] = d[X_cat].astype(object).fillna('Missing')

train_inner[X_cat].describe()

In [ ]:
# handle numerical missing values, then join together
#
# (1) Drop numeric features that are almost always missing in the TRAINING split (e.g. ANF/ANA is 99.9% missing):
#     with so little data a median fill means "everyone looks the same" and an indicator is nearly constant.
MAX_MISSING = 0.95
miss_train = train_inner[X_num].isna().mean()
X_num_kept = miss_train[miss_train <= MAX_MISSING].index.tolist()
print(f"Dropping {len(X_num) - len(X_num_kept)} of {len(X_num)} numeric columns that are >{MAX_MISSING:.0%} missing in training")

# (2) median fill + "was missing" indicator. FIT ON THE TRAINING SPLIT ONLY.
num_imputer = SimpleImputer(strategy='median', add_indicator=True)
num_imputer.fit(train_inner[X_num_kept])
imp_names = num_imputer.get_feature_names_out().tolist()    # original names, then 'missingindicator_<name>'

def impute_numeric(d):
    return pd.DataFrame(num_imputer.transform(d[X_num_kept]), columns=imp_names, index=d.index)

# (3) join numeric + binary + categorical
def build_frame(d):
    return pd.concat([impute_numeric(d), d[X_bin].fillna(0), d[X_cat]], axis=1)

Xtr, Xva, Xho = build_frame(train_inner), build_frame(val_inner), build_frame(heldout_df)
ytr, yva, yho = train_inner[OUTCOME_COL], val_inner[OUTCOME_COL], heldout_df[OUTCOME_COL]
print(Xtr.shape, Xva.shape, Xho.shape)

In [ ]:
# Make sure that imputer is learned / fit ONLY over your final frozen train split,
# and then applied to the test set (ie. if computing a mean, compute the mean of the training set)
# -> num_imputer.fit(...) above only ever saw train_inner. val_inner and heldout_df only went through .transform().
for name, d in [('train', Xtr), ('val', Xva), ('heldout', Xho)]:
    assert d.isnull().sum().sum() == 0, f"{name}: Data should not have missing values after imputation"
print("No missing values left in train / val / heldout.")

## Step 5: Build a simple baseline model 

In [ ]:
# First, preprocess the data, ie. imputation (done above), one hot encoding, scaling. Use sklearn.
ind_cols    = [c for c in imp_names if c.startswith('missingindicator_')]   # 0/1 "was missing" flags
num_scaled  = [c for c in imp_names if not c.startswith('missingindicator_')]

prep = ColumnTransformer([
    ('num',  StandardScaler(), num_scaled),                                    # continuous -> mean 0, sd 1
    ('cat',  OneHotEncoder(handle_unknown='ignore', sparse_output=False), X_cat),   # words -> 0/1 columns
    ('pass', 'passthrough', ind_cols + X_bin),                                 # already 0/1
], remainder='drop')

In [ ]:
# Define the model, XGBoost or Logistic Regression. Use sklearn.
# Logistic regression. The only knob is the regularization strength C (smaller C = stronger shrinkage),
# which we tune on the INTERNAL validation set (never on the held-out hospitals).
fitted = {}
for C in [0.001, 0.01, 0.1, 1.0]:
    m = Pipeline([('prep', clone(prep)), ('lr', LogisticRegression(C=C, max_iter=2000))])
    m.fit(Xtr, ytr)                                          # Learn on the training data
    fitted[C] = m
    print(f"C={C:<6} internal-val AUPRC = {average_precision_score(yva, m.predict_proba(Xva)[:, 1]):.4f}")

best_C = max(fitted, key=lambda C: average_precision_score(yva, fitted[C].predict_proba(Xva)[:, 1]))
model = fitted[best_C]
print("Chosen C:", best_C)

## Step 6: Evaluate model performance internally
Typical metrics for binary outcomes are AUROC, AUPRC, brier score, and accuracy (but ask yourself -- what do each metrics mean? what metrics do we care about most? what happens if Y is very skewed? is there one metric we should prioritize?) Note that for brier, lower brier is better. 

In [ ]:
def get_auroc(model, Xdata, Ydata):
    return roc_auc_score(Ydata, model.predict_proba(Xdata)[:, 1])

def get_auprc(model, Xdata, Ydata):
    return average_precision_score(Ydata, model.predict_proba(Xdata)[:, 1])

def get_brier(model, Xdata, Ydata):
    return brier_score_loss(Ydata, model.predict_proba(Xdata)[:, 1])

def score_all(y, p):
    # all metrics from labels y and predicted probabilities p (needs both classes in y)
    return dict(n=len(y), n_pos=int(np.sum(y)), prevalence=float(np.mean(y)),
                auroc=roc_auc_score(y, p), auprc=average_precision_score(y, p), brier=brier_score_loss(y, p))

In [ ]:
# calculate average, or perhospital internal model performance on the training data . basically you want a validation
# metric scores that are NOT the heldout hospitals. 
p_val = model.predict_proba(Xva)[:, 1]
internal = score_all(yva.to_numpy(), p_val)
int_auroc, int_auprc, int_brier = internal['auroc'], internal['auprc'], internal['brier']
print(f"INTERNAL (pooled validation): AUROC={int_auroc:.3f}  AUPRC={int_auprc:.3f}  Brier={int_brier:.4f}  "
      f"(prevalence={internal['prevalence']:.2%})")

# per training hospital (only hospitals with enough validation positives/negatives to score)
rows = []
for h in val_inner['hospitalid'].unique():
    m = (val_inner['hospitalid'] == h).to_numpy()
    y_h = yva.to_numpy()[m]
    if y_h.sum() >= 5 and (len(y_h) - y_h.sum()) >= 5:
        rows.append(dict(hospitalid=h, **score_all(y_h, p_val[m])))
internal_by_hosp = pd.DataFrame(rows)
print(f"Per-hospital internal metrics for {len(internal_by_hosp)} training hospitals "
      f"(mean AUROC={internal_by_hosp.auroc.mean():.3f}, mean AUPRC={internal_by_hosp.auprc.mean():.3f})")
internal_by_hosp.describe().T[['count', 'mean', 'std', 'min', 'max']]

**Which metric matters most?** *(draft, edit in your own words)*
- **AUROC**: if you pick one patient who died and one who survived, how often does the model rank the one who died as higher risk? 0.5 = coin flip. It ignores prevalence, so it can look good even when the outcome is very rare.
- **AUPRC**: of the patients the model flags as highest risk, how many really are positive (precision) and how many positives do we catch (recall)? It depends strongly on prevalence, so **always compare it to the prevalence baseline** (a useless model scores about the prevalence).
- **Brier score**: mean squared error of the predicted probabilities (lower is better). It checks whether the probabilities are *calibrated*, which AUROC/AUPRC do not.
- **Accuracy**: misleading when Y is skewed. If 4% die, predicting "nobody dies" is 96% accurate and useless.
- With a rare outcome I would prioritize **AUPRC (vs. prevalence) and AUROC**, and report Brier to check calibration.

## Step 7: Evaluate external model performance and also generalization gap 
NOTE: When developing our "method", we won't assume we know these heldout hospital data.

In [ ]:
# calculate heldout hospital performance. plot the worst. 
p_ho = model.predict_proba(Xho)[:, 1]
y_ho = yho.to_numpy()

rows = []
for h in sorted(heldout_df['hospitalid'].unique()):
    m = (heldout_df['hospitalid'] == h).to_numpy()
    rows.append(dict(hospitalid=h, hospital=str(h), **score_all(y_ho[m], p_ho[m])))
ext = pd.DataFrame(rows)

# generalization gap = internal - external  (for Brier, lower is better, so external - internal)
ext['gap_auroc'] = int_auroc - ext['auroc']
ext['gap_auprc'] = int_auprc - ext['auprc']
ext['gap_brier'] = ext['brier'] - int_brier
ext['auprc_over_prev'] = ext['auprc'] / ext['prevalence']     # AUPRC relative to chance, easier to compare across prevalences

ext.sort_values('gap_auroc', ascending=False).round(3)

In [ ]:
# FIGURE 1: AUROC and AUPRC at each held-out hospital, with the internal score as a dashed reference
order = ext.sort_values('auroc')['hospital'].tolist()                 # worst -> best AUROC
e = ext.set_index('hospital').loc[order].reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

sns.barplot(data=e, x='hospital', y='auroc', order=order, color='#4C72B0', ax=axes[0])
axes[0].axhline(int_auroc, ls='--', color='k', label=f'internal AUROC = {int_auroc:.3f}')
axes[0].set_ylim(max(0, e.auroc.min() - 0.1), 1)
axes[0].set_title('AUROC at each held-out hospital'); axes[0].set_ylabel('AUROC'); axes[0].set_xlabel('held-out hospital id')

sns.barplot(data=e, x='hospital', y='auprc', order=order, color='#55A868', ax=axes[1])
axes[1].axhline(int_auprc, ls='--', color='k', label=f'internal AUPRC = {int_auprc:.3f}')
axes[1].plot(range(len(e)), e['prevalence'], linestyle='none', marker='_', color='crimson', markersize=18,
             markeredgewidth=2.5, label='chance level (hospital prevalence)')
axes[1].set_title('AUPRC at each held-out hospital'); axes[1].set_ylabel('AUPRC'); axes[1].set_xlabel('held-out hospital id')

for ax in axes:
    ax.tick_params(axis='x', rotation=45)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.30), fontsize=9, frameon=False)   # legend below the plot
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_performance_by_heldout_hospital.png', dpi=200, bbox_inches='tight')
plt.show()

Gap > 0   -> model performs worse at this hospital

Gap ≈ 0   -> similar to internal validation

Gap < 0   -> model performs better at this hospital

In [ ]:
# define and plot the generalization gap using seaborn. 
order_gap = ext.sort_values('gap_auroc', ascending=False)['hospital'].tolist()   # biggest gap first
gap_long = ext.melt(id_vars='hospital', value_vars=['gap_auroc', 'gap_auprc'], var_name='metric', value_name='gap')
gap_long['metric'] = gap_long['metric'].map({'gap_auroc': 'AUROC gap', 'gap_auprc': 'AUPRC gap'})

# FIGURE 2: generalization gap by held-out hospital
plt.figure(figsize=(11, 4.5))
sns.barplot(data=gap_long, x='hospital', y='gap', hue='metric', order=order_gap, palette=['#4C72B0', '#55A868'])
plt.axhline(0, color='k', lw=1)
plt.title('Generalization gap (internal - held-out); positive = worse at the new hospital')
plt.xlabel('held-out hospital id'); plt.ylabel('gap'); plt.xticks(rotation=45)
plt.legend(title='')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_generalization_gap.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"Largest AUROC gap: hospital {order_gap[0]} ({ext.set_index('hospital').loc[order_gap[0], 'gap_auroc']:.3f}); "
      f"mean gap = {ext.gap_auroc.mean():.3f}, sd = {ext.gap_auroc.std():.3f}")

Note that if for the specific split you chose, the gap is 0, then 1) this is great!, and 2) any research into reducing the gap is not applicable. 
You can try different splits / Y / X  to see if you can find a split that has a non-zero gap.

## Step 8: Explore causes of gaps 
Once we have a X / Y / split setting where there is a large gap, can we ask: Why might some hospitals have larger gaps? Why might some splits induce larger gaps? 
Could this be from: 
1. Which features X we choose?
2. Differences in missingness between training and heldout? 
3. Differences in outcome Y prevalence? 
4. Differences in hospital characteristics?
5. .... other? 
You don't need to find an answer, there is likely no single cause, but exploring this is a good exercise. 

Bonus question: What role do we think the ML model itself has in generalization gaps??

In [ ]:
# For every held-out hospital: how different is it from the training hospitals?
#   miss_shift = mean |missing-rate(hospital) - missing-rate(train)| over numeric features
#   smd_shift  = mean |standardized mean difference| over numeric features  (feature-distribution shift)
#   prev_shift = hospital prevalence - training prevalence
tr_miss = train_inner[X_num_kept].isna().mean()
tr_mean = train_inner[X_num_kept].mean()
tr_std  = train_inner[X_num_kept].std().replace(0, np.nan)

rows, per_feature = [], {}
for h in ext['hospitalid']:
    d = heldout_df[heldout_df['hospitalid'] == h][X_num_kept]
    miss_diff = (d.isna().mean() - tr_miss)
    smd = (d.mean() - tr_mean) / tr_std
    per_feature[h] = pd.DataFrame({'miss_diff': miss_diff, 'smd': smd})
    hosp_prev = heldout_df.loc[heldout_df['hospitalid'] == h, OUTCOME_COL].mean()
    rows.append(dict(hospitalid=h, miss_shift=miss_diff.abs().mean(), smd_shift=smd.abs().mean(),
                     prev_shift=hosp_prev - ytr.mean()))
shift = pd.DataFrame(rows)
diag = ext.merge(shift, on='hospitalid')

# FIGURE 3: does the gap line up with how different the hospital is?
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
panels = [('miss_shift', 'missingness shift  (mean |Δ missing rate|)'),
          ('smd_shift',  'feature-distribution shift  (mean |std. mean diff|)'),
          ('prev_shift', 'outcome prevalence shift  (hospital - train)')]
for ax, (col, label) in zip(axes, panels):
    sns.regplot(data=diag, x=col, y='gap_auroc', ax=ax, ci=None, scatter_kws=dict(s=55), line_kws=dict(color='gray', lw=1.5))
    rho, pval = spearmanr(diag[col], diag['gap_auroc'])
    ax.set_title(f'Spearman ρ = {rho:.2f} (p = {pval:.2f})', fontsize=10)
    ax.set_xlabel(label); ax.set_ylabel('AUROC gap')
    for _, r in diag.iterrows():
        ax.annotate(r['hospital'], (r[col], r['gap_auroc']), fontsize=7, xytext=(3, 3), textcoords='offset points')
plt.suptitle(f'What is different about the hospitals with the biggest gap?  (n = {len(diag)} held-out hospitals)', y=1.03)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_gap_vs_shift_diagnostics.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Which FEATURES are risky? Features the model relies on heavily AND that look different at the worst hospital.
worst = ext.sort_values('gap_auroc', ascending=False).iloc[0]['hospitalid']
print("Worst held-out hospital:", worst)

coefs = pd.Series(model.named_steps['lr'].coef_[0], index=model.named_steps['prep'].get_feature_names_out())
coefs = coefs[coefs.index.str.startswith('num__')]
coefs.index = coefs.index.str.replace('num__', '', regex=False)

risk = (pd.DataFrame({'abs_coef': coefs.abs()})
        .join(per_feature[worst])
        .assign(abs_miss_diff=lambda t: t.miss_diff.abs(), abs_smd=lambda t: t.smd.abs()))
risk['risk_score'] = risk['abs_coef'] * (risk['abs_miss_diff'] + risk['abs_smd'])   # simple heuristic, not a theorem
risk.sort_values('risk_score', ascending=False).head(15).round(3)

**Your interpretation goes here.** Things to comment on after you run this on your data:
- Is the gap large or small overall (Figure 2)? Are a few hospitals clearly worse than the rest, or is it mostly noise from small samples (check `n` and `n_pos` in the table)?
- In Figure 3, does the gap track missingness shift, feature-distribution shift, or prevalence shift? With only ~10 held-out hospitals the correlations are weak evidence, so say so.
- In the last table, are the "risky" features (high weight in the model, very different at the worst hospital) things you could have flagged **without** seeing that hospital, using only the training hospitals? That is the question Project 2 is built on.
- Try another split (`CHOSEN_SPLIT = 'region=...'`) or another outcome and see whether the story changes.

**Bonus: role of the model itself.** A flexible model (e.g. XGBoost) can latch onto subtle hospital-specific patterns, such as which labs a hospital tends to order, which look predictive in training but do not transfer. A heavily regularized linear model is less able to do that, but may also leave signal on the table. Comparing the gaps of the two would be a natural experiment.